# Sequencing-anchored overall survival: text comparisons

Run after `1_data/03_prediction_datasets.ipynb` has built the sequencing-anchored prediction dataset and feature tables. Use a compute-backed Jupyter CPU session.

This notebook runs the existing Cox training modules for **`death_met:death`** (overall survival measured from sequencing date):

1. **Full cohort:** text plus baseline covariates versus the baseline model (gender, age at sequencing, cancer type).
2. **Common full-data cohort:** text versus stage, treatment, somatic, PRS, and metastatic burden. The training loader intersects the stage, treatment, somatic, and germline patient IDs before fitting *every* modality, including text. Metastatic burden is zero-filled and does not restrict membership.

Both analyses use the same sequencing anchor and endpoint, but their patient populations differ. Compare models **within** each table below; do not treat the full-cohort text estimate as the comparator for the modality table. Each runner writes its usual CV/test metrics; the modality runner also writes nested held-out risk scores. With `OVERWRITE = False`, completed outputs are reused.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config
from pipelines.training.run_feature_comp_task import ALL_MODALITIES
from pipelines.training.slurm_array_utils import _feature_file
from schemes import embedding_file, feature_held_out_dir, full_cohort_event_dir, scheme_results_dir

print(f"repo root: {REPO_ROOT}")
print(f"data root: {config.DATA_PATH}")
print(f"Python:    {sys.executable}")

## Configuration

The default hyperparameter grid and five-fold splits come from the training modules. `N_JOBS` is the number of CV jobs within a task; keep BLAS at one thread per job. Both tasks run sequentially, so this notebook does not compete with itself for cores or memory.

In [ ]:
SCHEME = "death_met"
EVENT = "death"
ANCHOR = "sequencing"

RUN_FULL = True
RUN_MODALITIES = True
OVERWRITE = False
N_JOBS = 4
MAX_ITER = 1000
BACKEND = "threading"

MODALITIES = ALL_MODALITIES  # stage, treatment, somatic, prs, text, metburden
FULL_DIR = Path(full_cohort_event_dir(SCHEME, EVENT, ANCHOR))
MODALITY_DIR = Path(scheme_results_dir(SCHEME, ANCHOR)) / "feature_comps" / EVENT
RISK_DIR = Path(feature_held_out_dir(SCHEME, EVENT, ANCHOR))

print(f"endpoint: {SCHEME}:{EVENT}; anchor: {ANCHOR}")
print(f"full-cohort output: {FULL_DIR}")
print(f"modality output:    {MODALITY_DIR}")
print(f"modalities: {', '.join(MODALITIES)}")

## Inputs and existing outputs

The full-cohort task needs the sequencing prediction parquet and cancer-type table. The common-cohort task also needs every modality table, even when some individual modality results already exist, because `--modality all` loads the common patient intersection before checking output files.

In [ ]:
PREDICTION_FILE = Path(config.SURV_PATH) / embedding_file(SCHEME, ANCHOR)
CANCER_FILE = Path(_feature_file("cancer_type_df.csv.gz", ANCHOR))
FULL_INPUTS = {
    "sequencing prediction dataset": PREDICTION_FILE,
    "sequencing cancer types": CANCER_FILE,
}
MODALITY_INPUTS = {
    **FULL_INPUTS,
    "sequencing cancer stage": Path(_feature_file("cancer_stage_df.csv.gz", ANCHOR)),
    "sequencing treatment": Path(_feature_file("categorical_treatment_data_by_line.csv.gz", ANCHOR)),
    "sequencing somatic": Path(_feature_file("complete_somatic_data_df.csv.gz", ANCHOR)),
    "germline / PRS": Path(config.FEATURE_PATH) / "complete_germline_data_df.csv.gz",
    "sequencing metastatic burden": Path(_feature_file("met_burden_df.csv.gz", ANCHOR)),
}


def missing_inputs(inputs: dict[str, Path]) -> list[str]:
    return [label for label, path in inputs.items() if not path.is_file()]


for label, path in MODALITY_INPUTS.items():
    print(f"[{'ok' if path.is_file() else 'MISSING':<7}] {label:<30} {path}")


def full_complete() -> bool:
    return all((FULL_DIR / f"{model}_{kind}.csv{suffix}").is_file()
               for model in ("text", "base")
               for kind, suffix in (("test", ""), ("val", ""), ("ipcw_reference", ".gz")))


def modality_complete(modality: str) -> bool:
    return all((MODALITY_DIR / f"{modality}_{kind}.csv{suffix}").is_file()
               for kind, suffix in (("test", ""), ("val", ""), ("ipcw_reference", ".gz"))) and (
        RISK_DIR / f"{modality}_risk_scores.csv").is_file()


print(f"\nFull-cohort outputs: {'complete' if full_complete() else 'pending'}")
for modality in MODALITIES:
    print(f"{modality:<10} {'complete' if modality_complete(modality) else 'pending'}")

## Fit full-cohort text and baseline models

Runs `run_full_cohort_event` for the sequencing anchor. It independently skips text or baseline when that model's test, validation, and IPCW files already exist.

In [ ]:
def run_module(module: str, args: list[str]) -> int:
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "OMP_NUM_THREADS": "1",
           "MKL_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "NUMEXPR_NUM_THREADS": "1"}
    cmd = [sys.executable, "-m", module, *args]
    print("Running:", " ".join(cmd), flush=True)
    start = time.perf_counter()
    completed = subprocess.run(cmd, cwd=REPO_ROOT, env=env)
    print(f"Exit {completed.returncode}; elapsed {(time.perf_counter() - start) / 60:.1f} min")
    return completed.returncode


COMMON_ARGS = ["--scheme", SCHEME, "--event", EVENT, "--anchor", ANCHOR,
               "--n-jobs", str(N_JOBS), "--max-iter", str(MAX_ITER), "--backend", BACKEND]
if OVERWRITE:
    COMMON_ARGS.append("--overwrite")

full_exit = None
if not RUN_FULL:
    print("Full-cohort run disabled.")
elif full_complete() and not OVERWRITE:
    print("Full-cohort text and baseline outputs already complete; reusing them.")
elif missing := missing_inputs(FULL_INPUTS):
    print("Full-cohort run blocked by missing inputs:", ", ".join(missing))
else:
    full_exit = run_module("pipelines.training.run_full_cohort_event", COMMON_ARGS)

print(f"Full-cohort outputs: {'complete' if full_complete() else 'incomplete'}")

## Fit text and every comparator on the common full-data cohort

`--modality all` loads the shared patient cohort once, then fits each modality separately. A failure in one modality is reported without hiding the others. Existing grid and risk-score outputs are skipped by the runner.

In [ ]:
modality_exit = None
if not RUN_MODALITIES:
    print("Modality run disabled.")
elif all(modality_complete(modality) for modality in MODALITIES) and not OVERWRITE:
    print("All six modality outputs already complete; reusing them.")
elif missing := missing_inputs(MODALITY_INPUTS):
    print("Modality run blocked by missing inputs:", ", ".join(missing))
else:
    modality_exit = run_module(
        "pipelines.training.run_feature_comp_task", [*COMMON_ARGS, "--modality", "all"]
    )

print("Modality outputs:")
for modality in MODALITIES:
    print(f"  {modality:<10} {'complete' if modality_complete(modality) else 'incomplete'}")

## Held-out test performance

The test CSVs report mean C-index and mean time-dependent AUC. Deltas in each table are `text − comparator` on that table's cohort. A missing row means its runner has not completed. These are descriptive held-out estimates, without a paired uncertainty calculation.

In [ ]:
import math
import polars as pl


def test_metrics(path: Path) -> tuple[float | None, float | None]:
    if not path.is_file():
        return None, None
    frame = pl.read_csv(path)
    if frame.is_empty():
        return None, None
    row = frame.row(0, named=True)
    values = []
    for key in ("mean_c_index", "mean_auc(t)"):
        value = row.get(key)
        value = float(value) if value is not None else None
        values.append(value if value is not None and math.isfinite(value) else None)
    return values[0], values[1]


def difference(text_value: float | None, other_value: float | None) -> float | None:
    return text_value - other_value if text_value is not None and other_value is not None else None


full_text = test_metrics(FULL_DIR / "text_test.csv")
full_base = test_metrics(FULL_DIR / "base_test.csv")
full_rows = [
    {"cohort": "full", "model": name, "c_index": score[0], "auc_t": score[1],
     "delta_c_index_vs_base": difference(score[0], full_base[0]),
     "delta_auc_t_vs_base": difference(score[1], full_base[1])}
    for name, score in (("base", full_base), ("text", full_text))
]

modality_scores = {m: test_metrics(MODALITY_DIR / f"{m}_test.csv") for m in MODALITIES}
common_text = modality_scores["text"]
modality_rows = [
    {"cohort": "common full-data", "model": m, "c_index": score[0], "auc_t": score[1],
     "delta_c_index_text_minus_model": difference(common_text[0], score[0]),
     "delta_auc_t_text_minus_model": difference(common_text[1], score[1])}
    for m, score in modality_scores.items()
]

with pl.Config(tbl_rows=20, tbl_cols=8, float_precision=4):
    print("Full cohort — text versus baseline")
    print(pl.DataFrame(full_rows))
    print("\nCommon full-data cohort — text versus each other modality")
    print(pl.DataFrame(modality_rows))

if not full_complete() or not all(modality_complete(m) for m in MODALITIES):
    print("\nSome outputs are incomplete; rerun the corresponding fit cell after resolving its logs or inputs.")